In [16]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('../..')))

In [17]:
from src.patient_data_dispatcher import PatientDataDispatcher
from src.core.enums import MileStone, DataFrequency, PatientDataType

import pandas as pd

import torch

In [18]:
dmo_features = [
    "wb_all_sum",
    "walkdur_all_sum",
    "wbsteps_all_sum",
    "wbdur_all_avg",
    "wbdur_all_p90",
    "wbdur_all_var",
    "cadence_all_avg",
    "strdur_all_avg",
    "cadence_all_var",
    "strdur_all_var",
    "ws_1030_avg",
    "strlen_1030_avg",
    "wb_10_sum",
    "ws_10_p90",
    "wb_30_sum",
    "ws_30_avg",
    "strlen_30_avg",
    "cadence_30_avg",
    "strdur_30_avg",
    "ws_30_p90",
    "cadence_30_p90",
    "ws_30_var",
    "strlen_30_var",
    "wb_60_sum",
    "total_worn_h",
]

In [19]:
pdd = PatientDataDispatcher(
    "../../config/config.yaml",
    dmo_features,
    MileStone.ALL,
    data_frequency=DataFrequency.WEEKLY,
    physical_subset=True,
)
ids = list(set(pdd.metadata["Local.Participant"].to_list()))
metadata = pdd.metadata

In [20]:
metadata

,Local.Participant,Participant.Site,Cohort,visit.number,visit.date,status.date,followup.date,proto_dev,proto_dev_err,CVS_C,...,ws_30_avg_w,strlen_30_avg_w,cadence_30_avg_w,strdur_30_avg_w,ws_30_max_w,cadence_30_max_w,ws_30_var_w,strlen_30_var_w,wb_60_sum_w,n_days_w
0,10376,CAU,MS,t1,2022-02-10 09:08:16,2022-02-10 09:08:16,2022-02-10 09:08:16,N,NaN,1,...,0.955,125.310,87.366,1.249,1.438,96.878,44.914,36.143,2.0,7.0
1,10376,CAU,MS,t2,2022-08-18 08:03:43,2022-08-18 08:03:43,2022-08-18 08:03:43,N,NaN,1,...,0.853,113.440,87.474,1.236,1.154,93.968,29.400,25.843,2.0,7.0
2,10376,CAU,MS,t3,2023-02-10 08:45:35,2023-02-10 08:45:35,2023-02-10 08:45:35,N,NaN,1,...,0.698,94.958,86.664,1.304,0.870,91.651,19.014,18.529,1.0,7.0
3,10376,CAU,MS,t4,2023-08-31 07:23:13,2023-08-31 07:23:13,2023-08-31 07:23:13,N,NaN,1,...,0.857,109.319,91.322,1.087,1.124,97.958,31.780,28.280,1.0,7.0
4,10376,CAU,MS,t5,2024-02-15 08:48:52,2024-02-15 08:48:52,2024-02-15 08:48:52,N,NaN,1,...,0.708,96.772,84.712,1.252,0.840,87.674,22.733,21.700,0.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2871,25238,USR,MS,t4,2024-03-28 15:43:12,2024-03-28 15:43:12,2024-03-28 15:43:12,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2872,25238,USR,MS,t5,2024-05-08 11:55:54,2024-05-08 11:55:54,2024-05-08 11:55:54,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2873,25239,USR,MS,t1,2022-10-05 10:03:11,2022-10-05 10:03:11,2022-10-05 10:03:11,N,NaN,1,...,0.848,102.646,97.804,1.175,1.008,108.685,16.700,9.129,18.0,7.0
2874,25239,USR,MS,t2,2023-03-02 00:00:00,2023-03-02 00:00:00,2023-03-02 00:00:00,N,NaN,1,...,0.841,101.950,97.977,1.207,0.953,106.482,12.125,7.425,20.0,4.0


In [21]:
# filter to specific columns
metadata = metadata[["Local.Participant", "pstatus", "wreason", "wreasonother", "EDFSCR1L"]]

In [22]:
# filter out attended patients
metadata = metadata[metadata["pstatus"] != "Attended"]

In [23]:
# count number of Excused patients
excused = metadata[metadata["pstatus"] == "Excused"]
excused

,Local.Participant,pstatus,wreason,wreasonother,EDFSCR1L
6,10377,Excused,NaN,NaN,NaN
126,10411,Excused,NaN,NaN,NaN
127,10411,Excused,NaN,NaN,NaN
156,10420,Excused,NaN,NaN,NaN
218,21146,Excused,NaN,NaN,NaN
...,...,...,...,...,...
2828,25228,Excused,NaN,NaN,NaN
2837,25230,Excused,NaN,NaN,NaN
2845,25232,Excused,NaN,NaN,NaN
2869,25238,Excused,NaN,NaN,NaN


In [24]:
withdrawn = metadata[metadata["pstatus"] == "Withdrawn"]
withdrawn

,Local.Participant,pstatus,wreason,wreasonother,EDFSCR1L
7,10377,Withdrawn,Personal reasons,NaN,NaN
92,10398,Withdrawn,Personal reasons,NaN,NaN
124,10410,Withdrawn,Other,"Moved to a different city in september, travel...",NaN
157,10420,Withdrawn,Other,The participant is annoyed by the clinic (medi...,NaN
233,21154,Withdrawn,Study visits too burdensome,NaN,NaN
...,...,...,...,...,...
2821,25226,Withdrawn,Study visits too burdensome,NaN,NaN
2831,25228,Withdrawn,Personal reasons,NaN,NaN
2839,25230,Withdrawn,Personal reasons,NaN,NaN
2843,25231,Withdrawn,Personal reasons,NaN,NaN


In [25]:
burdensom = withdrawn[withdrawn["wreason"] == "Study visits too burdensome"]
print(burdensom)
ids = set(burdensom["Local.Participant"].tolist())
ids


      Local.Participant    pstatus                      wreason wreasonother  \
233               21154  Withdrawn  Study visits too burdensome          NaN   
644               24125  Withdrawn  Study visits too burdensome          NaN   
985               24311  Withdrawn  Study visits too burdensome          NaN   
1027              24322  Withdrawn  Study visits too burdensome          NaN   
1077              24347  Withdrawn  Study visits too burdensome          NaN   
1397              24500  Withdrawn  Study visits too burdensome          NaN   
1771              25004  Withdrawn  Study visits too burdensome          NaN   
1794              25009  Withdrawn  Study visits too burdensome          NaN   
1913              25034  Withdrawn  Study visits too burdensome          NaN   
1952              25042  Withdrawn  Study visits too burdensome          NaN   
1955              25043  Withdrawn  Study visits too burdensome          NaN   
2081              25069  Withdrawn  Stud

{21154,
 24125,
 24311,
 24322,
 24347,
 24500,
 25004,
 25009,
 25034,
 25042,
 25043,
 25069,
 25072,
 25074,
 25100,
 25106,
 25107,
 25157,
 25225,
 25226,
 25239}

In [26]:
# find edss values for burdonsome patiants

edss = metadata[metadata["Local.Participant"].isin(ids)]
# edss = metadata["EDFSCR1L"]
edss

,Local.Participant,pstatus,wreason,wreasonother,EDFSCR1L
232,21154,Excused,NaN,NaN,NaN
233,21154,Withdrawn,Study visits too burdensome,NaN,NaN
644,24125,Withdrawn,Study visits too burdensome,NaN,NaN
985,24311,Withdrawn,Study visits too burdensome,NaN,NaN
1027,24322,Withdrawn,Study visits too burdensome,NaN,NaN
1077,24347,Withdrawn,Study visits too burdensome,NaN,NaN
1397,24500,Withdrawn,Study visits too burdensome,NaN,NaN
1771,25004,Withdrawn,Study visits too burdensome,NaN,NaN
1794,25009,Withdrawn,Study visits too burdensome,NaN,NaN
1912,25034,Excused,NaN,NaN,NaN


In [27]:
uncontactable = metadata[metadata["pstatus"] == "Uncontactable"]
uncontactable


,Local.Participant,pstatus,wreason,wreasonother,EDFSCR1L
214,21138,Uncontactable,NaN,NaN,NaN
215,21138,Uncontactable,NaN,NaN,NaN
220,21146,Uncontactable,NaN,NaN,NaN
237,21155,Uncontactable,NaN,NaN,NaN
241,21157,Uncontactable,NaN,NaN,NaN
...,...,...,...,...,...
2829,25228,Uncontactable,NaN,NaN,NaN
2830,25228,Uncontactable,NaN,NaN,NaN
2841,25231,Uncontactable,NaN,NaN,NaN
2842,25231,Uncontactable,NaN,NaN,NaN


In [28]:
# filter out NaN
metadata = metadata.dropna()
metadata

,Local.Participant,pstatus,wreason,wreasonother,EDFSCR1L
